## Creates two tables

### 1. tabel TRANSACTIONS_OBL_LOC

Loendab iga verbobl puhul unikaalseid root sõnu iga kohakäände jaoks.


### 2. tabel TRANSACTIONS_OBL_ACTOR_LOC_COUNTS

Loendab iga verbobl ja kohakäände puhul, kui palju on unikaalseid root sõnu, kui palju on märgitud elusateks ja kui palju on märgitud kohtadeks (nimekirja mitte verbi annotatsiooni põhjal).

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from common_sql import update_table, create_count_table_distinct, create_left_join_table
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [2]:
DB_DIR = "../example_data"

# transaktsioonide andmebaas
TRANSACTION_DB = f"{DB_DIR}/transactions.db"
# transaktsioonide andmebaas
ENRICHED_TRANSACTION_DB = f"{DB_DIR}/enriched_transactions.db"
# loodavad tabelid
PATTERN_MATCHES_DB = f"{DB_DIR}/pattern_matches.db"

TRANSACTION_HEAD = "transaction_head"
ENRICHED_TRANSACTIONS = "transaction_row"

# table for all obl locative case verbs and roots (maybe temp)
TRANSACTIONS_OBL_LOC = "trans_obl_loc"

# table for all obl locative case verbs and dictinct root counts
TRANSACTIONS_OBL_LOC_COUNTS = "trans_obl_loc_counts"

# table for elus and koht counts (verb+comp+case -> elus/koht/total distinct root count)
TRANSACTIONS_OBL_ACTOR_LOC_COUNTS = "trans_obl_actor_loc_counts"

# temporary count tables for all loc cases
t1, t2, t3, t4, t5, t6, t7 = "temp1", "temp2", "temp3", "temp4", "temp5", "temp6", "temp7"
# temporary verb table
verb = "verbs"
# temporary join tables for all loc cases
j0, j1, j2, j3, j4, j5, j6 = "join0", "join1", "join2", "join3", "join4", "join5", "join6"

# temporary tables for elus/koht count table
c1, c2, c3, c4 = "count1","count2","count3","count4",



## Connect to database

In [3]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS trans')
cur.execute(f'ATTACH DATABASE "{ENRICHED_TRANSACTION_DB}" AS entrans')

## Workflow

### Tabel 1

Võtta välja kõik mis on kohakäändes ja sõna deprel on obl

In [4]:
%%time

cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = TRANSACTIONS_OBL_LOC))

cur.execute("""
Create table {new_table} as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tbl1.verb_compound as verb_compound,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.form as root_form,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM {head_table} as tbl1
join entrans.{trans_table} as tr
on tbl1.id = tr.head_id
where tr.deprel = 'obl'
and 
(INSTR(',' || tr.feats || ',', ',' || 'abl' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'adit' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'all' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ad' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'el' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ill' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'in' || ',') > 0
)
""".format(new_table=TRANSACTIONS_OBL_LOC, head_table=TRANSACTION_HEAD, trans_table=ENRICHED_TRANSACTIONS))

CPU times: user 6.14 ms, sys: 963 µs, total: 7.1 ms
Wall time: 14.3 ms


#### tabelisse juurde veergu 'case', mis käändega on tegu

In [5]:
cur.execute("""ALTER TABLE {tbl} ADD loc_case VARCHAR(50)""".format(tbl=TRANSACTIONS_OBL_LOC))
con.commit()

update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'abl'", "INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0")
update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'adit'", "INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0")
update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'all'", "INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0")
update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'ad'", "INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0")
update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'el'", "INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0")
update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'ill'", "INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0")
update_table(con, TRANSACTIONS_OBL_LOC, "loc_case", "'in'", "INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0")


In [6]:
display_db_table(con, TRANSACTIONS_OBL_LOC)

,head_id,verb,verb_compound,transaction_id,root_word,root_form,word_deprel,pos,tr_feats,koht,elus,loc_case
0,2,toimuma,,1,lõpp,lõpus,obl,S,"com,in,sg",,,in
1,3,saama,pihta,7,keel,keeltele,obl,S,"all,com,pl",,,all
2,10,tulema,,19,sina,sul,obl,P,"ad,sg",,YES,ad
3,11,viilima,,22,tund,tundidest,obl,S,"com,el,pl",,,el
4,11,viilima,,23,juht,juhul,obl,S,"ad,com,sg",,YES,ad


### base tabel, kus on distinct verbid eelmisest tabelist ja iga kohakäände jaoks distinct root count veerg 

In [7]:
%%time

# create temp count tables for all cases

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t1, ["verb","verb_compound"], "root_word", "abl_cnt", 
                  "loc_case='abl'", ["verb", "verb_compound"])

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t2, ["verb","verb_compound"], "root_word", "adit_cnt", 
                  "loc_case='adit'", ["verb", "verb_compound"])

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t3, ["verb","verb_compound"], "root_word", "all_cnt", 
                  "loc_case='all'", ["verb", "verb_compound"])

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t4, ["verb","verb_compound"], "root_word", "ad_cnt", 
                  "loc_case='ad'", ["verb", "verb_compound"])

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t5, ["verb","verb_compound"], "root_word", "el_cnt", 
                  "loc_case='el'", ["verb", "verb_compound"])

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t6, ["verb","verb_compound"], "root_word", "ill_cnt", 
                  "loc_case='ill'", ["verb", "verb_compound"])

create_count_table_distinct(con, TRANSACTIONS_OBL_LOC, t7, ["verb","verb_compound"], "root_word", "in_cnt", 
                  "loc_case='in'", ["verb", "verb_compound"])

# create temp join tables

cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = verb))

cur.execute("""
CREATE TABLE {count_tbl} AS select verb, verb_compound from {tbl2}
""".format(count_tbl=verb, tbl2 = TRANSACTIONS_OBL_LOC))

create_left_join_table(con, source_tbl1=verb, source_tbl2=t1,result_table=j0,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j0, source_tbl2=t2,result_table=j1,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j1, source_tbl2=t3,result_table=j2,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j2, source_tbl2=t4, result_table=j3,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j3, source_tbl2=t5,result_table=j4,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt", "el_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j4, source_tbl2=t6, result_table=j5,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt", "el_cnt", "ill_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

create_left_join_table(con, source_tbl1=j5, source_tbl2=t7, result_table=j6,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "abl_cnt", "adit_cnt", "all_cnt", "ad_cnt", "el_cnt", "ill_cnt", "in_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound")

CPU times: user 8.51 ms, sys: 8.39 ms, total: 16.9 ms
Wall time: 74.9 ms


In [8]:
%%time

# join all necessary tables
cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = TRANSACTIONS_OBL_LOC_COUNTS))

cur.execute("""
CREATE TABLE {count_tbl} AS select * from {join_tbl}
""".format(count_tbl=TRANSACTIONS_OBL_LOC_COUNTS, join_tbl=j6))

update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "abl_cnt", 0, "abl_cnt is null")
update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "adit_cnt",0, "adit_cnt is null")
update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "all_cnt", 0, "all_cnt is null")
update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "ad_cnt", 0, "ad_cnt is null")
update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "el_cnt", 0, "el_cnt is null")
update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "ill_cnt", 0, "ill_cnt is null")
update_table(con, TRANSACTIONS_OBL_LOC_COUNTS, "in_cnt", 0, "in_cnt is null")


CPU times: user 5.4 ms, sys: 5.15 ms, total: 10.5 ms
Wall time: 39.6 ms


In [9]:
# delete temporary tables
for tbl in [j0, j1, j2, j3, j4, j5, j6, t1, t2, t3, t4, t5, t6, t7, verb]:
    cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = tbl))

In [11]:
display_db_table(con, TRANSACTIONS_OBL_LOC_COUNTS)

,verb,verb_compound,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,toimuma,,0,0,0,0,0,0,1
1,saama,pihta,0,0,1,0,0,0,0
2,tulema,,0,3,1,4,0,0,0
3,viilima,,0,0,0,1,1,0,0
4,viilima,,0,0,0,1,1,0,0


## 2. tabel

### base tabel kus on verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [12]:
%%time

# total distinct root count

cur.execute("""DROP table if exists {tbl}""".format(tbl=c1))

cur.execute("""
CREATE TABLE {new_table} AS
select verb, verb_compound, loc_case, count(distinct root_word) as root_cnt
from {verbtable}
group by verb, verb_compound, loc_case
""".format(new_table=c1, verbtable=TRANSACTIONS_OBL_LOC))


# elus dictinct root count

cur.execute("""DROP table if exists {tbl}""".format(tbl=c2))

cur.execute("""
CREATE TABLE {new_table} AS
select  verb, verb_compound, loc_case, count(distinct root_word) as elus_cnt
from {verbtable}
where elus='YES'
group by verb, verb_compound, loc_case
""".format(new_table=c2,verbtable=TRANSACTIONS_OBL_LOC))


# koht distinct root count

cur.execute("""DROP table if exists {tbl}""".format(tbl=c3))

cur.execute("""
CREATE TABLE {new_table} AS
select  verb, verb_compound, loc_case, count(distinct root_word) as koht_cnt
from {verbtable}
where koht='YES'
group by verb, verb_compound, loc_case
""".format(new_table=c3,verbtable=TRANSACTIONS_OBL_LOC))

CPU times: user 4.13 ms, sys: 3.21 ms, total: 7.33 ms
Wall time: 15.4 ms


In [13]:
# join count tables

create_left_join_table(con, source_tbl1=c1, source_tbl2=c2, result_table=c4,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "tbl1.loc_case", "elus_cnt", "root_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound and tbl1.loc_case = tbl2.loc_case")

create_left_join_table(con, source_tbl1=c4, source_tbl2=c3, result_table=TRANSACTIONS_OBL_ACTOR_LOC_COUNTS,
                selected_columns=["tbl1.verb", "tbl1.verb_compound", "tbl1.loc_case", "tbl1.elus_cnt","koht_cnt", "tbl1.root_cnt"],
                condition="tbl1.verb=tbl2.verb and tbl1.verb_compound=tbl2.verb_compound and tbl1.loc_case = tbl2.loc_case")


In [14]:
# set null values to 0
update_table(con, TRANSACTIONS_OBL_ACTOR_LOC_COUNTS, "elus_cnt", 0, "elus_cnt is null")
update_table(con, TRANSACTIONS_OBL_ACTOR_LOC_COUNTS, "koht_cnt", 0, "koht_cnt is null")

In [15]:
# delete temporary tables
for tbl in [c1, c2, c3, c4]:
    cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = tbl))

In [16]:
display_db_table(con, TRANSACTIONS_OBL_ACTOR_LOC_COUNTS, 10, 'head')

,verb,verb_compound,loc_case,elus_cnt,koht_cnt,root_cnt
0,aitama,,ad,1,0,1
1,ajama,,adit,1,0,1
2,algama,,el,0,0,1
3,andma,,ad,0,0,1
4,andma,,all,1,0,1
5,andma,,in,0,0,2
6,andma,alla,all,1,0,1
7,arvama,,el,0,0,1
8,astuma,üles,ad,0,0,1
9,baseeruma,,ad,0,0,1


In [19]:
#example result with more data 
#query = """select * from {tbl} order by root_cnt desc""".format(tbl = TRANSACTIONS_OBL_ACTOR_LOC_COUNTS)
#source = pd.read_sql_query(query, con)
#source

,verb,verb_compound,kaane,elus_cnt,koht_cnt,root_cnt
0,saama,,el,1274,402,19632
1,andma,,all,1601,250,11612
2,rääkima,,el,802,202,10891
3,saama,,in,134,306,8532
4,tulema,,ad,1134,166,8468
...,...,...,...,...,...,...
74715,šveitsima,,el,0,0,1
74716,švipsima,,ad,0,0,1
74717,žestikuleerima,,ad,0,1,1
74718,žisraelima,,ad,0,0,1


In [45]:
# save the table to csv if necessary
#source.to_csv(TRANSACTIONS_OBL_ACTOR_LOC_COUNTS+'.csv', index=False, sep=",", encoding="utf-8")

In [7]:
con.close()